import yfinance as yf  
import pandas as pd  
import numpy as np  
from datetime import datetime    
   
ticker = yf.Ticker("SPY")  
   
// Spot price  
spot = ticker.history(period="1d")["Close"].iloc[-1]  
spot_date = ticker.history(period="1d").index[-1].strftime("%Y-%m-%d")  
print(f"SPY spot: ${spot:.2f} ({spot_date})")  
    
// Select expiration dates (mix of short/mid/long term)  
all_expiries = ticker.options   
today = datetime.now().date()   
   
short, mid, long_ = [], [], []   
for exp_str in all_expiries:   
    days = (datetime.strptime(exp_str, "%Y-%m-%d").date() - today).days   
    if days < 7:   
        continue   
    elif days <= 45:   
        short.append(exp_str)   
    elif days <= 180:   
        mid.append(exp_str)   
    else:   
        long_.append(exp_str)   
   
def pick(lst, n):      
    if len(lst) <= n:      
        return lst   
    idx = np.linspace(0, len(lst) - 1, n, dtype=int)   
    return [lst[i] for i in idx]   
   
selected = pick(short, 2) + pick(mid, 2) + pick(long_, 2)   
   
print(f"\nSelected expiries ({len(selected)}):")   
for e in selected:   
    days = (datetime.strptime(e, "%Y-%m-%d").date() - today).days   
    print(f"  {e}  ({days} days)")   
   
// Fetch options chains   
all_data = []   
for expiry in selected:   
    chain = ticker.option_chain(expiry)   
    for opt_type, df in [("call", chain.calls), ("put", chain.puts)]:   
        df = df.copy()   
        df["option_type"] = opt_type   
        df["expiry"] = expiry   
        days = (datetime.strptime(expiry, "%Y-%m-%d").date() - today).days   
        df["T"] = days / 365.0   
        df["S"] = spot   
        df["spot_date"] = spot_date   
        all_data.append(df)   
   
options_df = pd.concat(all_data, ignore_index=True)   
   
cols = [   
    "expiry", "option_type", "strike", "lastPrice", "bid", "ask",   
    "volume", "openInterest", "impliedVolatility", "T",   
    "S", "spot_date", "contractSymbol",   
]   
cols = [c for c in cols if c in options_df.columns]   
options_df = options_df[cols]   
   
options_df.to_csv("spy_options_raw.csv", index=False)   


In [2]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline

df = pd.read_csv("data/spy_options.csv")

print("Task 1.2: Options Data")
print("Ticker: SPY (SPDR S&P 500 ETF Trust)")
print(f"Total options: {len(df)}")
print(f"Calls: {(df['option_type'] == 'call').sum()}")
print(f"Puts: {(df['option_type'] == 'put').sum()}")
print(f"Expirations: {df['expiry'].nunique()}")
print(f"Strike range: ${df['strike'].min():.0f} – ${df['strike'].max():.0f}")
print(f"T range: {df['T'].min():.4f} – {df['T'].max():.4f} years")

print("\n Per-expiry breakdown:")
print(f"{'Expiry':<14} {'Calls':>5} {'Puts':>5} {'Total':>6}  {'T (years)':>10}")

for exp in sorted(df['expiry'].unique()):
    sub = df[df['expiry'] == exp]
    c = (sub['option_type'] == 'call').sum()
    p = (sub['option_type'] == 'put').sum()
    print(f"  {exp:<14} {c:>5} {p:>5} {len(sub):>6}  {sub['T'].iloc[0]:>10.4f}")

S = df['S'].iloc[0]
spot_date = df['spot_date'].iloc[0]

print("\n Task 1.3: Market Data")
print(f"Spot price: ${S:.2f}")
print(f"Spot date: {spot_date}")


treasury_date = "2026-02-09"

treasury_data = {
    # tenor_years: yield_percent
    1/12:    3.72,   # 1 Month
    1.5/12:  3.72,   # 1.5 Months
    2/12:    3.73,   # 2 Months
    3/12:    3.69,   # 3 Months
    4/12:    3.70,   # 4 Months
    6/12:    3.59,   # 6 Months
    1:       3.43,   # 1 Year
    2:       3.48,   # 2 Years
    3:       3.56,   # 3 Years
    5:       3.75,   # 5 Years
    7:       3.97,   # 7 Years
    10:      4.22,   # 10 Years
    20:      4.79,   # 20 Years
    30:      4.85,   # 30 Years
}

tenors = np.array(list(treasury_data.keys()))
yields_pct = np.array(list(treasury_data.values()))
yields_dec = yields_pct / 100.0  # convert to decimal

print("\nRisk-free rate source: U.S. Treasury Par Yield Curve")
print(f"Treasury date: {treasury_date}")
print(f"\n{'Tenor':<12} {'Yield':>8}")
labels = ['1 Mo','1.5 Mo','2 Mo','3 Mo','4 Mo','6 Mo',
          '1 Yr','2 Yr','3 Yr','5 Yr','7 Yr','10 Yr','20 Yr','30 Yr']
for label, t, y in zip(labels, tenors, yields_pct):
    print(f"{label:<12} {y:>7.3f}%")

cs = CubicSpline(tenors, yields_dec, bc_type='natural')

def get_risk_free_rate(T):
    """Interpolate risk-free rate for time-to-expiry T (years)."""
    T_clipped = np.clip(T, tenors.min(), tenors.max())
    return float(cs(T_clipped))

df['r'] = df['T'].apply(get_risk_free_rate)

print("\nInterpolated rates for each expiry:")
print(f"{'Expiry':<14} {'T (years)':>10} {'r':>8}")
print(f"{'-'*14} {'-'*10} {'-'*8}")
for exp in sorted(df['expiry'].unique()):
    T_val = df[df['expiry'] == exp]['T'].iloc[0]
    r_val = df[df['expiry'] == exp]['r'].iloc[0]
    print(f"{exp:<14} {T_val:>10.4f} {r_val*100:>7.3f}%")


output_path = "data/spy_options_enriched.csv"
df.to_csv(output_path, index=False)

print("Enriched dataset saved to spy_options_enriched.csv")
print(f"Columns: {list(df.columns)}")
print(f"Rows: {len(df)}")

Task 1.2: Options Data
Ticker: SPY (SPDR S&P 500 ETF Trust)
Total options: 1568
Calls: 801
Puts: 767
Expirations: 6
Strike range: $50 – $1350
T range: 0.0192 – 2.8438 years

 Per-expiry breakdown:
Expiry         Calls  Puts  Total   T (years)
  2026-02-18        67    67    134      0.0192
  2026-03-27        53    97    150      0.1205
  2026-03-31       272   246    518      0.1315
  2026-07-31       114   127    241      0.4658
  2026-09-18       120   101    221      0.6000
  2028-12-15       175   129    304      2.8438

 Task 1.3: Market Data
Spot price: $692.12
Spot date: 2026-02-10

Risk-free rate source: U.S. Treasury Par Yield Curve
Treasury date: 2026-02-09

Tenor           Yield
1 Mo           3.720%
1.5 Mo         3.720%
2 Mo           3.730%
3 Mo           3.690%
4 Mo           3.700%
6 Mo           3.590%
1 Yr           3.430%
2 Yr           3.480%
3 Yr           3.560%
5 Yr           3.750%
7 Yr           3.970%
10 Yr          4.220%
20 Yr          4.790%
30 Yr         

In [4]:
df = pd.read_csv('data/spy_options_enriched.csv')
df.head()

,expiry,option_type,strike,lastPrice,bid,ask,volume,openInterest,impliedVolatility,T,S,spot_date,contractSymbol,r
0,2026-02-18,call,600.0,80.03,91.73,94.50,NaN,2,0.567875,0.019178,692.119995,2026-02-10,SPY260218C00600000,0.0372
1,2026-02-18,call,625.0,64.92,66.79,69.55,2.0,2,0.534795,0.019178,692.119995,2026-02-10,SPY260218C00625000,0.0372
2,2026-02-18,call,645.0,42.68,46.89,49.65,1.0,1,0.413458,0.019178,692.119995,2026-02-10,SPY260218C00645000,0.0372
3,2026-02-18,call,650.0,36.03,41.95,44.71,2.0,2,0.383795,0.019178,692.119995,2026-02-10,SPY260218C00650000,0.0372
4,2026-02-18,call,660.0,34.58,32.16,34.92,2.0,3,0.326057,0.019178,692.119995,2026-02-10,SPY260218C00660000,0.0372


In [5]:
df.tail()

,expiry,option_type,strike,lastPrice,bid,ask,volume,openInterest,impliedVolatility,T,S,spot_date,contractSymbol,r
1563,2028-12-15,put,915.0,217.61,220.0,224.98,NaN,0,0.097070,2.843836,692.119995,2026-02-10,SPY281215P00915000,0.035473
1564,2028-12-15,put,1000.0,306.54,305.0,309.98,NaN,0,0.120797,2.843836,692.119995,2026-02-10,SPY281215P01000000,0.035473
1565,2028-12-15,put,1005.0,311.54,310.0,314.98,NaN,0,0.122094,2.843836,692.119995,2026-02-10,SPY281215P01005000,0.035473
1566,2028-12-15,put,1340.0,646.51,645.0,649.98,NaN,0,0.193459,2.843836,692.119995,2026-02-10,SPY281215P01340000,0.035473
1567,2028-12-15,put,1350.0,657.15,655.0,659.98,30.0,0,0.195229,2.843836,692.119995,2026-02-10,SPY281215P01350000,0.035473
